# Module 13 Lab - Building ML Pipelines**Objective:** To understand the importance of `scikit-learn` **Pipelines** for creating robust, reproducible, and professional machine learning workflows.**In this lab, you will refactor code from a previous lab into a clean, professional `Pipeline` object.**

## Part 1: Why Use Pipelines?**Concept:** As you've seen, a typical ML workflow involves multiple steps: loading data, cleaning it, splitting it, preprocessing features (scaling, encoding), and finally, training a model. Managing all these steps separately can be messy and error-prone.**Data Leakage:** A major risk of manual preprocessing is **data leakage**. This happens when information from the test set accidentally "leaks" into the training process. For example, if you calculate the mean for scaling using the *entire* dataset before splitting, the model has already "seen" the test data, leading to overly optimistic performance estimates.**A `scikit-learn` Pipeline solves these problems by:**1.  **Encapsulating** all workflow steps into a single object.2.  **Preventing Data Leakage:** It ensures that preprocessing steps are fitted *only* on the training data during cross-validation or when calling `.fit()`.3.  **Improving Reproducibility:** The entire workflow is saved as one object, making it easy to reuse and deploy.

## Part 2: The "Manual" Way (What We Did Before)Let's revisit the Titanic dataset and the steps we took to prepare the data and train a model. This code should look familiar. Notice how many separate objects and steps there are.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load data
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# Basic feature engineering and cleaning
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df.drop('Cabin', axis=1, inplace=True)

X = df.drop(['Survived', 'Name', 'Ticket', 'PassengerId'], axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify feature types
numeric_features = ['Age', 'Fare', 'SibSp', 'Parch']
categorical_features = ['Pclass', 'Sex', 'Embarked']

# Manual Preprocessing
scaler = StandardScaler()
X_train_scaled_num = scaler.fit_transform(X_train[numeric_features])
X_test_scaled_num = scaler.transform(X_test[numeric_features]) # Note: using .transform() here!

encoder = OneHotEncoder(handle_unknown='ignore')
X_train_encoded_cat = encoder.fit_transform(X_train[categorical_features])
X_test_encoded_cat = encoder.transform(X_test[categorical_features])

# Combine preprocessed features
X_train_processed = np.hstack((X_train_scaled_num, X_train_encoded_cat.toarray()))
X_test_processed = np.hstack((X_test_scaled_num, X_test_encoded_cat.toarray()))

# Train model
model = RandomForestClassifier(random_state=42)
model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
print(f"Accuracy (Manual Method): {accuracy_score(y_test, y_pred):.2%}")

/tmp/ipykernel_2395/1649700768.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
/tmp/ipykernel_2395/1649700768.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

Accuracy (Manual Method): 82.68%


## Part 3: The "Pipeline" WayNow, let's do the exact same thing but encapsulate all the preprocessing steps into a single `Pipeline`.**Your Task:** Use `make_pipeline` and `make_column_transformer` to build a complete workflow. This is the modern, professional way to build models in `scikit-learn`.

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer # Added missing import
from sklearn.preprocessing import StandardScaler, OneHotEncoder # Added missing imports
from sklearn.ensemble import RandomForestClassifier # Added missing import
from sklearn.model_selection import train_test_split # Added missing import
from sklearn.metrics import accuracy_score # Added missing import
import pandas as pd # Added missing import
import numpy as np # Added missing import

# Reload the data to start fresh
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
df.drop(['Cabin', 'Name', 'Ticket', 'PassengerId'], axis=1, inplace=True)
X = df.drop('Survived', axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- ENTER YOUR CODE HERE ---
# 1. Create a pipeline for numeric features
#    This pipeline will first impute missing 'Age' values with the median, then scale the features.
numeric_transformer = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler()
)
# 2. Create a pipeline for categorical features
#    This pipeline will first impute missing 'Embarked' values with the most frequent value, then one-hot encode.
categorical_transformer = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore')
)
# 3. Use ColumnTransformer to apply different transformers to different columns
preprocessor = make_column_transformer(
    (numeric_transformer, ['Age', 'Fare', 'SibSp', 'Parch']),
    (categorical_transformer, ['Pclass', 'Sex', 'Embarked'])
)
# 4. Create the final, full pipeline
#    This chains the preprocessor and the final model together.
final_pipeline = make_pipeline(
    preprocessor,
    RandomForestClassifier(random_state=42)
)
# 5. Fit and evaluate the entire pipeline in one step!
final_pipeline.fit(X_train, y_train)
y_pred_pipeline = final_pipeline.predict(X_test)
print(f"Accuracy (Pipeline Method): {accuracy_score(y_test, y_pred_pipeline):.2%}")

Accuracy (Pipeline Method): 82.68%


## 📝 Reflective Knowledge Check
**Instructions:** Answer the following questions in this markdown cell.
1.  **Code Comparison:** Look at the "Manual Way" versus the "Pipeline Way". What are the three biggest advantages you see in using the Pipeline approach?
2.  **Data Leakage Explained:** In the manual code, we used `scaler.fit_transform()` on the training data but only `scaler.transform()` on the test data. Why was this distinction crucial? How does the Pipeline automatically handle this for you?
3.  **Extending the Pipeline:** Imagine you wanted to add a PCA step to reduce dimensionality *after* scaling and encoding but *before* the RandomForestClassifier. How would you modify your `final_pipeline` object to include this step? (You don't need to write the full code, just describe where you would add `PCA()`.)
4.  **Real-World Value:** You are handing your model over to another team to deploy into a web application. Why is giving them the single `final_pipeline` object much safer and more reliable than giving them the 5 separate objects (`scaler`, `encoder`, `model`, etc.) from the manual approach?

**[ENTER YOUR ANSWERS HERE]**

1.  **Code Comparison Advantages:**
    *   **Cleaner Code:** The Pipeline approach makes the code much shorter and easier to read because it groups many steps together into one object.
    *   **Less Error-Prone:** When you do things manually, it's easy to forget a step or apply a transformation incorrectly. The Pipeline handles this automatically, reducing mistakes.
    *   **Easier to Manage:** All the preprocessing and the model are bundled into one object, making it simpler to save, load, and use the whole workflow.

2.  **Data Leakage Explained:**
    *   **Why the distinction was crucial:** When we use `scaler.fit_transform()` on the training data, the `scaler` learns the mean and standard deviation *only* from the training data. We then use `scaler.transform()` on the test data to apply those *same* learned values. If we used `fit_transform()` on the test data as well, the scaler would learn from the test data, which would be like the model "peeking" at the test data before being evaluated. This would lead to an overly optimistic performance estimate because the model indirectly learned from the test data's distribution.
    *   **How the Pipeline automatically handles this for you:** The Pipeline's `fit()` method is called only on the training data. During this `fit()` call, all `fit_transform()` operations within the pipeline (like for the `StandardScaler` or `OneHotEncoder`) are applied *only* to the training data. When `predict()` or `transform()` is called on new data (like the test set), the pipeline automatically uses the `transform()` method for all preprocessing steps, applying the transformations learned *only* from the training data. This completely prevents data leakage.

3.  **Extending the Pipeline:**
    *   To add a PCA step, I would insert `PCA()` as a new step in the `final_pipeline` object. It would go after the `preprocessor` (which handles scaling and encoding) and before the `RandomForestClassifier`. So, the structure would be `make_pipeline(preprocessor, PCA(), RandomForestClassifier())`.

4.  **Real-World Value:**
    *   Giving them the single `final_pipeline` object is much safer and more reliable because it ensures consistency and prevents errors. All the necessary data preprocessing steps (imputation, scaling, encoding) and the final model are encapsulated in one self-contained unit. This means the other team doesn't have to manually apply each preprocessing step in the correct order or with the correct parameters, reducing the chance of human error or subtle discrepancies that could lead to incorrect model predictions or unexpected behavior in a production environment. It guarantees that the model receives data in the exact same format it was trained on.

## Reflective Journal

This lab was a great introduction to machine learning pipelines in `scikit-learn`. Initially, seeing the "Manual Way" with all the separate steps for data loading, cleaning, splitting, preprocessing (scaling, encoding), and model training felt a bit overwhelming. There were many individual objects like `scaler` and `encoder` to manage. Understanding the risk of "data leakage" if preprocessing is not done carefully, especially regarding the training and testing sets, was a crucial lesson. It highlighted why manually applying `fit_transform` and `transform` separately is so important.

Moving on to the "Pipeline Way" was a real eye-opener. The `make_pipeline` and `make_column_transformer` functions dramatically simplified the code. Instead of juggling multiple steps and objects, everything was neatly packaged into a single `final_pipeline` object. This makes the code much cleaner and easier to follow, which is a huge advantage for someone just starting out. The fact that the pipeline automatically handles the `fit_transform` and `transform` logic for training and test data, preventing data leakage, is a massive relief and feels much safer.

Finally, seeing the feature importances was very insightful. It's cool how the pipeline not only automates the workflow but also allows us to easily inspect the underlying model components, like the `RandomForestClassifier`, to understand what features are driving its predictions. This holistic approach, from raw data to a trained model and then feature insights, all within a streamlined pipeline, really drives home the value of using these tools for building robust and understandable machine learning solutions. It feels like a solid step towards more professional and reproducible ML workflows.